# LocateAnything — batched/compiled benchmark

One notebook to run the full benchmark suite and produce `results_<tag>.json`.
Set `BATCH_SIZE` below, run all cells; then feed the result(s) to
`combine_results.ipynb` -> `EVIDENCE.md`.

- The **model** is cloned from `batched-vectorized-decode` and overlaid onto the HF
  snapshot, so the run measures that branch's own code.
- The pristine baseline is loaded *inside* the benchmark (the stacked/compile tasks), so the
  speedups are vs the unmodified model.

**Code changes, exact diffs, testing methodology, and per-change speed are all in
`EVIDENCE.md`** — this notebook is just setup + run.

## 0 · Config — everything you'd tweak, in one place

Edit this cell, then **Run all**. Every knob the run cell needs lives here and is passed
through unchanged — nothing to hunt for further down.

In [ ]:
# === CONFIG — edit, then Run all ===
from types import SimpleNamespace
CFG = SimpleNamespace(
    branch       = "batched-vectorized-decode",
    tag          = "batched",
    batch_size   = 8,                        # rerun at 16/32/... (same runtime, see the switch cell) for the compile/stacked batch-size curve
    n_limit      = 50,                       # images per detection dataset (eval / compile tasks)
    datasets     = ("COCO", "LVIS"),         # detection mAP datasets; grouping uses RefCOCOg (pinned in the grounded task)
    tasks        = ("eval", "grounded", "speed", "profile",
                    "compile_probe", "compile_loop", "compile_ap", "stacked_eval"),
    evaldata     = "/content/EvalData",      # EvalData root (download dest + image root)
)   # setup adds CFG.commit; the run writes back CFG.run_tag / CFG.out

## 1 · Setup — clone tooling + model, prepare the runtime

In [ ]:
import os, sys, getpass, logging, subprocess, shutil
logging.basicConfig(level=logging.INFO, format="%(message)s")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"   # fork lacks upstream LFS blobs; code only
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
REPO = "/content/repo"
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "transformers==4.57.1", "tokenizers==0.22.0", "accelerate",
                "peft==0.12.0", "timm", "huggingface_hub", "pandas", "tqdm",
                "matplotlib", "decord", "lmdb", "requests"], check=False)
    tok = getpass.getpass("GitHub token (Contents: read): ").strip()
    if os.path.isdir(os.path.join(REPO, "Embodied")):
        pass  # already cloned
    else:
        if os.path.exists(REPO):
            shutil.rmtree(REPO)
        res = subprocess.run(
            ["git", "clone", "-b", CFG.branch, f"https://{tok}@github.com/semajyllek/Eagle.git", REPO],
            capture_output=True, text=True, env={**os.environ})
        if res.returncode != 0 or not os.path.isdir(os.path.join(REPO, "Embodied")):
            raise RuntimeError("clone failed:\n%s" % (res.stderr or res.stdout).replace(tok, "***"))
else:
    REPO = subprocess.run(["git", "rev-parse", "--show-toplevel"],
                          capture_output=True, text=True).stdout.strip()
sys.path.insert(0, os.path.join(REPO, "Embodied"))      # repro tooling
MODEL_LOCANY = os.path.join(REPO, "Embodied", "eaglevl", "utils", "locany")
CFG.model_branch = CFG.branch
CFG.commit = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
import torch
RUN_GPU = torch.cuda.is_available()
logging.getLogger(__name__).info("tag=%s  branch=%s@%s  GPU=%s", CFG.tag, CFG.branch, CFG.commit, RUN_GPU)

## Code changes & deep dive

The exact code changes (motivation + diffs + how-tested + per-change speed), the full
correctness methodology (CPU tests, semantic parity, the mAP gate), and all measured
performance live in **`EVIDENCE.md`**.

## 2 · Run the benchmark (GPU) and save the record

Loads the branch's code overlaid onto the snapshot and runs the shared suite, all via
the real shipped API (`worker.predict` / `worker.predict_batch`): detection (batched-vs-B=1
+ parity + length buckets), grounded (one-image-many-queries), the speed
sweep, the per-step profile, the compile probes (feasibility, real-loop A/B, mAP gate vs
eager), and the stacked ladder (B=1-pristine -> batched -> batched+compiled, vs the
unmodified model). Writes `results_<tag>_bs<N>.json`.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
if RUN_GPU:
    from repro import load_overlaid_worker, run_benchmark
    from repro.combine import (detection_table, length_buckets_table,
                               grounded_table, speed_plot, profile_table,
                               compile_probe_table, compile_loop_table,
                               compile_ap_table, stacked_eval_table)
    worker, snap = load_overlaid_worker(src_dir=MODEL_LOCANY)
    rec = run_benchmark(worker, CFG)
    res = {rec["tag"]: rec}
    print("Detection (batched vs B=1):"); display(Markdown(detection_table(res)))
    print("Throughput sweep (sequential vs batched):"); display(Markdown(speed_plot(res)))
    display(Markdown(length_buckets_table(res)))
    print("One image, many queries:"); display(Markdown(grounded_table(res)))
    print("Decode-loop bookkeeping profile:"); display(Markdown(profile_table(res)))
    print("torch.compile feasibility probe:"); display(Markdown(compile_probe_table(res)))
    print("Real decode-loop torch.compile A/B:"); display(Markdown(compile_loop_table(res)))
    print("Compile mAP gate:"); display(Markdown(compile_ap_table(res)))
    print("Real-eval-workload stacked speedup (vs pristine):"); display(Markdown(stacked_eval_table(res)))
    try:
        from google.colab import files; files.download(CFG.out)
    except Exception as e:
        print("download skipped:", e)
else:
    print("No GPU — run on a GPU runtime to produce the record.")

## 3 · Visualize parity (optional, post-hoc — no effect on the benchmark)

Draws the detected boxes + `<ref>` labels **side by side** for two passes, reading the
prediction files the run already wrote (`compile_ap`'s `preds_eager.jsonl` vs
`preds_<compiled>.jsonl`). This touches *no* timed code — pure post-processing of saved
outputs. `only_diffs=True` shows only the images where eager and compiled disagree at the
detection gate (the near-ties behind the ~1% mAP delta); flip it to `False` to eyeball
matched detections. Point it at `preds_before.jsonl`/`preds_after.jsonl` for B=1-vs-batched.

In [ ]:
import os
_d = os.path.join(CFG.evaldata, "compile_ap_out", CFG.datasets[0])   # compile_ap wrote preds_<tag>.jsonl here
if os.path.exists(os.path.join(_d, "preds_eager.jsonl")):
    from repro.viz import parity_overlay
    import matplotlib.pyplot as plt
    fig = parity_overlay(os.path.join(_d, "preds_eager.jsonl"),
                         os.path.join(_d, "preds_default_dyn.jsonl"),
                         image_root=CFG.evaldata, labels=("eager", "compiled"),
                         n=4, only_diffs=False)     # only_diffs=True -> just the disagreements
    if fig is not None:
        plt.show()
else:
    print("Run the benchmark first — compile_ap writes the preds this reads.")

## 🔄 Switch batch size — keep the runtime (same GPU)

Run this cell, then edit the **CONFIG** cell (`batch_size`) and re-run setup
+ run. Keeps the exact GPU (compile/stacked numbers vary by hardware, so same-GPU comparison
matters) and the cached weights (no 7 GB re-download). `repro.reset_env` frees the model, clears
the CUDA cache, and purges the cached `trust_remote_code` modules — that purge is the key step:
the overlay overwrites the snapshot's code, but `trust_remote_code` caches the *imported* module,
so without it you'd silently keep running stale code.

In [ ]:
from repro import reset_env
print(reset_env(globals(), model_dir="/content/repo" if IN_COLAB else None))